# nexora_stage_runner

Job entry point for Nexora's long, model, data and Fabric operations
(fabric-hosting.md, CONTRACTS.md#fabric-operations). `operation = "smoke"` keeps
T-003's and T-005's acceptance criteria working; any other `operation` (for
example `"POST /runs/{id}/compile"`) goes through `app.fabric.dispatch` (T-102)
via `app.fabric.job.run_job`, which records `job_instance_id`, takes the run's
lease, dispatches as this job's own user, and writes `runs/<run_id>/jobs/<job_id>.json`.

In [ ]:
job_id = ""
run_id = ""
operation = "smoke"
payload = "{}"

In [ ]:
import json

args = json.loads(payload) if isinstance(payload, str) else (payload or {})

if operation == "smoke":
    from app.tools.smoke import run_job as run_smoke_checks

    result = run_smoke_checks(only=args.get("only"))
else:
    from app.fabric.job import run_job

    result = run_job(job_id, run_id, operation, args)

# T-704 FR-5: the exit value carries run_id and operation too, so the Monitoring
# hub's job instance entry can be matched back to the run without opening it.
exit_value = {"run_id": run_id, "operation": operation, **result}

try:
    import notebookutils

    notebookutils.notebook.exit(json.dumps(exit_value))
except ImportError:
    print(json.dumps(exit_value, indent=2))